In [1]:
import regex as re

# 示例 1: split_by_special_tokens 函数
def split_by_special_tokens(text, special_tokens):
    if not special_tokens:
        return [text]
    escaped = [re.escape(t) for t in special_tokens]
    escaped.sort(key=len, reverse=True)  # 长的先匹配
    pattern = '|'.join(escaped)
    parts = re.split(f'({pattern})', text)
    return [p for p in parts if p]

# 测试
text = "Hello <|endoftext|> world <|endoftext|>!"
special_tokens = ["<|endoftext|>"]
parts = split_by_special_tokens(text, special_tokens)
print("分割结果:", parts)

# 示例 2: GPT-2 风格的 pre-tokenization
GPT2_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

for part in parts:
    if part in special_tokens:
        print(f"Special token: {part}")
    else:
        tokens = re.findall(GPT2_PATTERN, part)
        print(f"Pre-tokenized: {tokens}")

# 示例 3: 使用 finditer（内存友好）
print("\n使用 finditer:")
for part in parts:
    if part not in special_tokens:
        for match in re.finditer(GPT2_PATTERN, part):
            token = match.group()
            print(f"Token: '{token}'")

分割结果: ['Hello ', '<|endoftext|>', ' world ', '<|endoftext|>', '!']
Pre-tokenized: ['Hello', ' ']
Special token: <|endoftext|>
Pre-tokenized: [' world', ' ']
Special token: <|endoftext|>
Pre-tokenized: ['!']

使用 finditer:
Token: 'Hello'
Token: ' '
Token: ' world'
Token: ' '
Token: '!'


In [3]:
PAT = r"\p{L}+|\p{N}+|[^\s\p{L}\p{N}]+"  # regex 支持 \p{L}
pat = re.compile(PAT, flags=re.VERSION1)  # 预编译
text = "Hello 123 你好！"
# findall 一次返回所有
print(pat.findall(text))
# finditer 逐个处理（内存友好）
for m in pat.finditer(text):
    print(m.group())

['Hello', '123', '你好', '！']
Hello
123
你好
！


In [5]:

s = "aXbXc"
print(re.split(r"(X)", s))      # ['a', 'X', 'b', 'X', 'c']  捕获组 => 保留分隔符
print(re.split(r"(?:X)", s))    # ['a', 'b', 'c']            非捕获组 => 不保留

['a', 'X', 'b', 'X', 'c']
['a', 'b', 'c']


In [6]:
special_tokens = ["<|endoftext|>", "<|start|>"]
escaped = [re.escape(t) for t in special_tokens]
escaped.sort(key=len, reverse=True)            # 长的先匹配，防止子串问题
pattern = "|".join(escaped)                    # e.g. "<\\|endoftext\\|>|<\\|start\\|>"
parts = re.split(f"({pattern})", text)      # 捕获组 => 保留 special token

In [10]:
type(f"({pattern})"),type(rf"({pattern})")

(str, str)

In [11]:
import regex
# 前瞻：匹配数字后面紧跟 "kg" 的数字
print(regex.findall(r"\d+(?=kg)", "5kg 10m 20kg"))  # ['5', '20']
# 后顾：匹配出现在 "$" 之后的数字（假设固定长度）
print(regex.findall(r"(?<=\$)\d+", "$100 and $20"))  # ['100', '20']

['5', '20']
['100', '20']


In [12]:
import regex
s = "<p>first</p><p>second</p>"
print(regex.findall(r"<p>.*</p>", s))   # 贪婪：会匹配整个字符串
print(regex.findall(r"<p>.*?</p>", s))  # 非贪婪：分别匹配两个 <p> 块

['<p>first</p><p>second</p>']
['<p>first</p>', '<p>second</p>']


In [15]:
phone = "yi13812345678er"
new_phone = re.sub(r"(\d{3})\d{4}(\d{4})", r"\1****\2", phone)
new_phone

'yi138****5678er'

In [16]:
# 演示 sub 中匹配 vs 捕获组的区别

import regex as re

phone = "13812345678"
pattern = r"(\d{3})\d{4}(\d{4})"

# 1. 分析匹配和捕获
match = re.search(pattern, phone)
if match:
    print("整个匹配:", match.group(0))  # 整个匹配对象
    print("捕获组1:", match.group(1))   # 第一个捕获组
    print("捕获组2:", match.group(2))   # 第二个捕获组

# 2. sub 替换整个匹配，但可以用捕获组构建新字符串
new_phone = re.sub(pattern, r"\1****\2", phone)
print("替换结果:", new_phone)

# 3. 对比：如果替换字符串不使用捕获组
new_phone2 = re.sub(pattern, "****", phone)  # 替换整个匹配为 "****"
print("替换整个匹配:", new_phone2)

# 4. 多个匹配的情况
text = "联系方式：13812345678 或 13987654321"
result = re.sub(pattern, r"\1****\2", text)
print("多匹配替换:", result)

整个匹配: 13812345678
捕获组1: 138
捕获组2: 5678
替换结果: 138****5678
替换整个匹配: ****
多匹配替换: 联系方式：138****5678 或 139****4321


# Regex 中匹配 vs 捕获组的区别

## 核心概念

### 匹配 (Match)
- **整个匹配对象**：正则表达式匹配到的完整字符串
- `match.group(0)` 或 `match.group()` 返回整个匹配

### 捕获组 (Capture Group)
- **子匹配**：用 `()` 包围的部分
- `match.group(1)`, `match.group(2)` 等返回对应的捕获组
- 在 `sub()` 中用 `\1`, `\2` 等引用

### sub() 的工作原理
```python
re.sub(pattern, replacement, text)
```
- **替换对象**：整个匹配 (group(0))
- **替换内容**：可以用 `\1`, `\2` 等引用捕获组来构建新字符串

## 示例分析

```python
phone = "13812345678"
pattern = r"(\d{3})\d{4}(\d{4})"  # 匹配整个手机号
replacement = r"\1****\2"         # 用捕获组构建新字符串

result = re.sub(pattern, replacement, phone)
# 结果: "138****5678"
```

**过程**：
1. `(\d{3})` 匹配 "138" → 捕获组1
2. `\d{4}` 匹配 "1234" → 不捕获
3. `(\d{4})` 匹配 "5678" → 捕获组2
4. **替换整个匹配** "13812345678" 为 `\1****\2` = "138****5678"

## match/search vs sub

### match/search
```python
match = re.search(pattern, text)
match.group(0)  # 整个匹配
match.group(1)  # 捕获组1
```

### sub
```python
re.sub(pattern, replacement, text)
# replacement 中可以用 \1, \2 引用捕获组
# 但替换的是整个匹配对象
```

In [19]:
it = re.finditer(r"1([a-z])([0-9])", "1a1b2")
for m in it:
    print("full:", m.group(0), "g1:", m.group(1), "g2:", m.group(2))
    # full: a1 g1: a g2: 1

full: 1a1 g1: a g2: 1


In [20]:
# raw 字符串只影响字面量解析
s1 = r"\n"   # 两个字符 '\' 和 'n'
s2 = "\\n"   # 等价，也表示 '\' 和 'n'
s3 = "\n"    # 实际换行
print(s1, len(s1))   # 输出: \n 2
print(s2, len(s2))   # 输出: \n 2
print(repr(s3), len(s3))  # 输出: '\n' 1

\n 2
\n 2
'\n' 1


In [21]:
import regex as re
token = "<|endoftext|>"

# 错（不要直接把 token 拼进 pattern）：
bad_pattern = f"({token})"   # 结果会把 '|' 解释为 alternation
print("bad_pattern:", bad_pattern)

# 正确：先转义，再拼接
escaped = re.escape(token)   # -> '<\\|endoftext\\|>'
pattern = f"({escaped})"
print("escaped pattern:", pattern)

text = "Hello <|endoftext|> world"
print(re.split(pattern, text))  # 捕获组保留 special token

bad_pattern: (<|endoftext|>)
escaped pattern: (<\|endoftext\|>)
['Hello ', '<|endoftext|>', ' world']


In [22]:
# rf'...' 可以同时是 raw 和 f-string
escaped = re.escape("<|a|>")
pat1 = f"({escaped})"   # 普通 f-string，插值后得到转义文本
pat2 = rf"({escaped})"  # 同样有效 —— raw 只影响字面量解析，不影响插值结果
print(pat1 == pat2)     # True

True
